In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [34]:
df=pd.read_csv(r'/content/sample_data/WHO COVID-19 cases.csv')
vac=pd.read_csv(r'/content/vaccinations.csv')
df.info()
df.describe()
vac.info()
vac.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58560 entries, 0 to 58559
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Date_reported      58560 non-null  object 
 1   Country_code       58316 non-null  object 
 2   Country            58560 non-null  object 
 3   Continent          58560 non-null  object 
 4   WHO_region         54168 non-null  object 
 5   New_cases          39260 non-null  float64
 6   Cumulative_cases   58560 non-null  int64  
 7   New_deaths         25084 non-null  float64
 8   Cumulative_deaths  58560 non-null  int64  
dtypes: float64(2), int64(2), object(5)
memory usage: 4.0+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 196246 entries, 0 to 196245
Data columns (total 16 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   location                             196246 non-null

,total_vaccinations,people_vaccinated,people_fully_vaccinated,total_boosters,daily_vaccinations_raw,daily_vaccinations,total_vaccinations_per_hundred,people_vaccinated_per_hundred,people_fully_vaccinated_per_hundred,total_boosters_per_hundred,daily_vaccinations_per_million,daily_people_vaccinated,daily_people_vaccinated_per_hundred
count,8.541700e+04,8.113200e+04,7.806100e+04,5.360000e+04,7.097100e+04,1.950290e+05,85417.000000,81132.000000,78061.000000,53600.000000,195029.000000,1.921770e+05,192177.000000
mean,5.616980e+08,2.487064e+08,2.286639e+08,1.505811e+08,7.398640e+05,2.838758e+05,124.279558,53.501409,48.680182,36.301489,1851.477596,1.060707e+05,0.074980
std,1.842160e+09,8.006461e+08,7.403763e+08,4.360697e+08,3.183064e+06,1.922352e+06,85.098042,29.379655,29.042282,30.218208,3117.828731,7.866884e+05,0.176216
min,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000
25%,1.970788e+06,1.050009e+06,9.644000e+05,6.022820e+05,2.010000e+03,2.790000e+02,44.770000,27.880000,21.220000,5.920000,106.000000,4.300000e+01,0.001000
50%,1.439435e+07,6.901088e+06,6.191345e+06,5.765440e+06,2.053100e+04,3.871000e+03,130.550000,64.300000,57.920000,35.905000,605.000000,7.710000e+02,0.014000
75%,1.161972e+08,5.093295e+07,4.773185e+07,4.019072e+07,1.736115e+05,3.180300e+04,194.990000,77.780000,73.610000,57.620000,2402.000000,9.307000e+03,0.073000
max,1.357877e+10,5.631264e+09,5.177943e+09,2.817381e+09,4.967320e+07,4.369181e+07,410.230000,129.070000,126.890000,150.470000,117113.000000,2.107127e+07,11.711000


In [37]:

import pandas as pd

# 1. Make copies
cases = df.copy()
vax = vac.copy()

# 2. Convert dates
cases["Date_reported"] = pd.to_datetime(
    cases["Date_reported"], dayfirst=True, errors="coerce"
)

vax["date"] = pd.to_datetime(
    vax["date"], errors="coerce"
)

# 3. Standardize country names
cases["Country"] = cases["Country"].str.strip()
vax["location"] = vax["location"].str.strip()

# 4. Select required columns
cases = cases[
    [
        "Date_reported", "Country_code", "Country",
        "Continent", "WHO_region", "New_cases",
        "Cumulative_cases", "New_deaths",
        "Cumulative_deaths"
    ]
]

vax = vax[
    [
        "location", "iso_code", "date",
        "daily_vaccinations", "total_vaccinations",
        "people_vaccinated", "people_fully_vaccinated",
        "total_boosters"
    ]
]

# 5. Rename vaccination columns
vax = vax.rename(columns={
    "location": "Country",
    "iso_code": "Country_code",
    "date": "Date_reported"
})

print("Cases dataset:", cases.shape)
print("Vaccination dataset:", vax.shape)
print("Cases date range:",
      cases["Date_reported"].min(), "to",
      cases["Date_reported"].max())
print("Vaccination date range:",
      vax["Date_reported"].min(), "to",
      vax["Date_reported"].max())

# Convert dates using explicit formats
cases["Date_reported"] = pd.to_datetime(
    cases["Date_reported"],
    format="%d-%m-%Y",
    errors="coerce"
)

vax["Date_reported"] = pd.to_datetime(
    vax["Date_reported"],
    format="%Y-%m-%d",
    errors="coerce"
)

print("Cases invalid dates:", cases["Date_reported"].isna().sum())
print("Vaccination invalid dates:", vax["Date_reported"].isna().sum())


Cases dataset: (58560, 9)
Vaccination dataset: (196246, 8)
Cases date range: 2020-01-03 00:00:00 to 2024-12-05 00:00:00
Vaccination date range: 2020-12-02 00:00:00 to 2024-08-14 00:00:00
Cases invalid dates: 35280
Vaccination invalid dates: 0


/tmp/ipykernel_1863/1057629058.py:12: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  vax["date"] = pd.to_datetime(


In [39]:

print("Missing vaccination values:")
print(
    covid_final[
        [
            "total_vaccinations",
            "people_vaccinated",
            "people_fully_vaccinated"
        ]
    ].isna().sum()
)

print("\nVaccination coverage by country — latest available record:")
latest_vax = (
    covid_final.sort_values("Date_reported")
    .dropna(subset=["people_vaccinated"])
    .groupby("Country", as_index=False)
    .tail(1)
    .sort_values("people_vaccinated", ascending=False)
)

display(
    latest_vax[
        [
            "Country", "Date_reported",
            "people_vaccinated",
            "people_fully_vaccinated",
            "total_vaccinations"
        ]
    ].head(10)
)

# Export combined dataset
covid_final.to_csv(
    "covid_cases_deaths_vaccinations.csv",
    index=False
)
print("\nCombined dataset exported successfully.")


Missing vaccination values:
total_vaccinations         48586
people_vaccinated          49046
people_fully_vaccinated    49840
dtype: int64

Vaccination coverage by country — latest available record:


,Country,Date_reported,people_vaccinated,people_fully_vaccinated,total_vaccinations
10364,China,2022-03-27,1.275541e+09,1.240777e+09,3.254984e+09
23541,India,2022-04-03,9.848381e+08,8.282295e+08,1.834501e+09
55993,United States of America,2022-04-03,2.553624e+08,2.174990e+08,5.601818e+08
23785,Indonesia,2022-04-03,1.962409e+08,1.588305e+08,3.771089e+08
6949,Brazil,2022-04-03,1.810781e+08,1.602729e+08,4.135596e+08
40130,Pakistan,2022-03-13,1.280741e+08,1.018812e+08,2.193686e+08
4021,Bangladesh,2022-04-03,1.275441e+08,1.077127e+08,2.436427e+08
27201,Japan,2022-04-03,1.024675e+08,1.006337e+08,2.543456e+08
34277,Mexico,2022-04-03,8.558029e+07,7.971176e+07,1.919079e+08
44037,Russian Federation,2022-04-03,7.995475e+07,7.284123e+07,1.636012e+08



Combined dataset exported successfully.


In [41]:

# 1. Check dataset size
print("Dataset shape:", covid_final.shape)

# 2. Check missing values
print("\nMissing values in each column:")
print(covid_final.isnull().sum())

# 3. Check duplicate rows
print("\nComplete duplicate rows:", covid_final.duplicated().sum())

# 4. Check duplicate country-date records
print(
    "\nDuplicate country-date records:",
    covid_final.duplicated(
        subset=["Country_code", "Date_reported"]
    ).sum()
)

# 5. Check missing-value percentages
print("\nMissing value percentage:")
print((covid_final.isnull().mean() * 100).round(2).sort_values(ascending=False))


Dataset shape: (58560, 17)

Missing values in each column:
Date_reported                              0
Country_code                             244
Country                                    0
Continent                                  0
WHO_region                              4392
New_cases                              19300
Cumulative_cases                           0
New_deaths                             33476
Cumulative_deaths                          0
Country_standard                           0
total_vaccinations                     48586
people_vaccinated                      49046
people_fully_vaccinated                49840
daily_vaccinations                     46593
total_vaccinations_per_hundred         48586
people_vaccinated_per_hundred          49046
people_fully_vaccinated_per_hundred    49840
dtype: int64

Complete duplicate rows: 0

Duplicate country-date records: 0

Missing value percentage:
people_fully_vaccinated_per_hundred    85.11
people_fully_vaccinated     

In [43]:

# Make a backup before cleaning
covid_df = covid_final.copy()

# Fill missing country codes from records of the same country
country_code_map = (
    covid_df.groupby("Country_standard")["Country_code"].first()
)

covid_df["Country_code"] = covid_df["Country_code"].fillna(
    covid_df["Country_standard"].map(country_code_map)
)

# Fill missing WHO regions from records of the same country
region_map = (
    covid_df.groupby("Country_standard")["WHO_region"].first()
)

covid_df["WHO_region"] = covid_df["WHO_region"].fillna(
    covid_df["Country_standard"].map(region_map)
)

# Check remaining missing values
print("Missing values after filling country information:")
print(covid_df.isnull().sum())

print("\nDataset shape:", covid_df.shape)

print(
    "\nDuplicate country-date records:",
    covid_df.duplicated(
        subset=["Country_code", "Date_reported"]
    ).sum()
)


Missing values after filling country information:
Date_reported                              0
Country_code                             244
Country                                    0
Continent                                  0
WHO_region                              4392
New_cases                              19300
Cumulative_cases                           0
New_deaths                             33476
Cumulative_deaths                          0
Country_standard                           0
total_vaccinations                     48586
people_vaccinated                      49046
people_fully_vaccinated                49840
daily_vaccinations                     46593
total_vaccinations_per_hundred         48586
people_vaccinated_per_hundred          49046
people_fully_vaccinated_per_hundred    49840
dtype: int64

Dataset shape: (58560, 17)

Duplicate country-date records: 0


In [45]:

import numpy as np
import pandas as pd

# Convert blank strings and common missing-value markers to NaN
covid_final = covid_final.replace(
    r"^\s*$",
    np.nan,
    regex=True
)

covid_final = covid_final.replace(
    ["NA", "N/A", "null", "None"],
    np.nan
)

# Confirm missing values after standardization
print(covid_final.isnull().sum())


Date_reported                              0
Country_code                             244
Country                                    0
Continent                                  0
WHO_region                              4392
New_cases                              19300
Cumulative_cases                           0
New_deaths                             33476
Cumulative_deaths                          0
Country_standard                           0
total_vaccinations                     48586
people_vaccinated                      49046
people_fully_vaccinated                49840
daily_vaccinations                     46593
total_vaccinations_per_hundred         48586
people_vaccinated_per_hundred          49046
people_fully_vaccinated_per_hundred    49840
dtype: int64


In [46]:

# Show countries with missing country codes
print("Countries with missing country codes:")
display(
    covid_df.loc[
        covid_df["Country_code"].isna(),
        ["Country", "Country_standard", "Continent"]
    ].drop_duplicates().sort_values("Country")
)

# Show countries with missing WHO regions
print("\nCountries with missing WHO regions:")
display(
    covid_df.loc[
        covid_df["WHO_region"].isna(),
        ["Country", "Country_standard", "Continent"]
    ].drop_duplicates().sort_values("Country")
)


Countries with missing country codes:


,Country,Country_standard,Continent
36356,Namibia,Namibia,Africa



Countries with missing WHO regions:


,Country,Country_standard,Continent
16592,Falkland Islands (Malvinas),Falkland Islands,South America
16836,Faroe Islands,Faeroe Islands,Europe
17812,French Guiana,French Guiana,North America
19520,Gibraltar,Gibraltar,Africa
20496,Guadeloupe,Guadeloupe,North America
21228,Guernsey,Guernsey,Europe
22448,Holy See,Holy See,Europe
26108,Isle of Man,Isle of Man,Europe
27328,Jersey,Jersey,Europe
30744,Liechtenstein,Liechtenstein,Europe


In [47]:

def most_common_valid(series):
    values = series.dropna()
    if values.empty:
        return np.nan
    return values.mode().iloc[0]

country_code_map = (
    covid_df.groupby("Country_standard")["Country_code"]
    .agg(most_common_valid)
)

region_map = (
    covid_df.groupby("Country_standard")["WHO_region"]
    .agg(most_common_valid)
)

covid_df["Country_code"] = covid_df["Country_code"].fillna(
    covid_df["Country_standard"].map(country_code_map)
)

covid_df["WHO_region"] = covid_df["WHO_region"].fillna(
    covid_df["Country_standard"].map(region_map)
)

print("Missing country codes:", covid_df["Country_code"].isna().sum())
print("Missing WHO regions:", covid_df["WHO_region"].isna().sum())


Missing country codes: 244
Missing WHO regions: 4392


In [48]:

# Fill Namibia's ISO country code
covid_df.loc[
    covid_df["Country_standard"].eq("Namibia")
    & covid_df["Country_code"].isna(),
    "Country_code"
] = "NA"

# Verified WHO region mappings for the listed countries/territories
who_region_mapping = {
    "Falkland Islands": "AMRO",
    "Faeroe Islands": "EURO",
    "French Guiana": "AMRO",
    "Gibraltar": "EURO",
    "Guadeloupe": "AMRO",
    "Guernsey": "EURO",
    "Holy See": "EURO",
    "Isle of Man": "EURO",
    "Jersey": "EURO",
    "Liechtenstein": "EURO",
    "Martinique": "AMRO",
    "Mayotte": "AFRO",
    "Pitcairn": "WPRO",
    "Réunion": "AFRO",
    "Saint Barthélemy": "AMRO",
    "Saint Helena": "AFRO",
    "Saint Martin (French part)": "AMRO",
    "Saint Pierre and Miquelon": "AMRO"
}

# Fill only missing WHO regions
for country, region in who_region_mapping.items():
    mask = (
        covid_df["Country_standard"].eq(country)
        & covid_df["WHO_region"].isna()
    )
    covid_df.loc[mask, "WHO_region"] = region

# Check results
print("Missing country codes:", covid_df["Country_code"].isna().sum())
print("Missing WHO regions:", covid_df["WHO_region"].isna().sum())


Missing country codes: 0
Missing WHO regions: 0


In [49]:

# Check actual country names for missing country codes
print("Missing country-code records:")
print(
    covid_df.loc[
        covid_df["Country_code"].isna(),
        ["Country", "Country_standard"]
    ].drop_duplicates().to_string(index=False)
)

# Check the exact standardized names for missing WHO regions
print("\nMissing WHO-region country names:")
print(
    covid_df.loc[
        covid_df["WHO_region"].isna(),
        ["Country", "Country_standard"]
    ].drop_duplicates().to_string(index=False)
)

# Check whether the intended mapping names exist in the data
print("\nCheck Namibia:")
print(covid_df["Country_standard"].eq("Namibia").sum())

print("\nCheck WHO mapping matches:")
for country in [
    "Falkland Islands", "Faeroe Islands", "French Guiana",
    "Gibraltar", "Guadeloupe", "Guernsey", "Holy See",
    "Isle of Man", "Jersey", "Liechtenstein", "Martinique",
    "Mayotte", "Pitcairn", "Réunion", "Saint Barthélemy",
    "Saint Helena", "Saint Martin (French part)",
    "Saint Pierre and Miquelon"
]:
    print(country, ":", covid_df["Country_standard"].eq(country).sum())


Missing country-code records:
Empty DataFrame
Columns: [Country, Country_standard]
Index: []

Missing WHO-region country names:
Empty DataFrame
Columns: [Country, Country_standard]
Index: []

Check Namibia:
244

Check WHO mapping matches:
Falkland Islands : 244
Faeroe Islands : 244
French Guiana : 244
Gibraltar : 244
Guadeloupe : 244
Guernsey : 244
Holy See : 244
Isle of Man : 244
Jersey : 244
Liechtenstein : 244
Martinique : 244
Mayotte : 244
Pitcairn : 244
Réunion : 244
Saint Barthélemy : 244
Saint Helena : 244
Saint Martin (French part) : 244
Saint Pierre and Miquelon : 244


In [50]:

print("Dataset shape:", covid_df.shape)

missing_summary = pd.DataFrame({
    "Missing_Count": covid_df.isnull().sum(),
    "Missing_Percentage": (
        covid_df.isnull().mean() * 100
    ).round(2)
})

display(
    missing_summary.sort_values(
        "Missing_Count",
        ascending=False
    )
)


Dataset shape: (58560, 17)


,Missing_Count,Missing_Percentage
people_fully_vaccinated_per_hundred,49840,85.11
people_fully_vaccinated,49840,85.11
people_vaccinated,49046,83.75
people_vaccinated_per_hundred,49046,83.75
total_vaccinations,48586,82.97
total_vaccinations_per_hundred,48586,82.97
daily_vaccinations,46593,79.56
New_deaths,33476,57.17
New_cases,19300,32.96
Date_reported,0,0.00


In [51]:

# Keep an unchanged copy of the cleaned source data
covid_clean = covid_df.copy()

# Keep missing observations as NaN
# Do not fill unknown case, death, or vaccination values with zero

# Check available observations
check_cols = [
    "New_cases",
    "New_deaths",
    "total_vaccinations",
    "people_vaccinated",
    "people_fully_vaccinated",
    "daily_vaccinations"
]

print("Available observations:")
print(covid_clean[check_cols].notna().sum())

print("\nMissing observations:")
print(covid_clean[check_cols].isna().sum())


Available observations:
New_cases                  39260
New_deaths                 25084
total_vaccinations          9974
people_vaccinated           9514
people_fully_vaccinated     8720
daily_vaccinations         11967
dtype: int64

Missing observations:
New_cases                  19300
New_deaths                 33476
total_vaccinations         48586
people_vaccinated          49046
people_fully_vaccinated    49840
daily_vaccinations         46593
dtype: int64


In [52]:

print("Negative new-case records:",
      (covid_clean["New_cases"] < 0).sum())

print("Negative new-death records:",
      (covid_clean["New_deaths"] < 0).sum())

display(
    covid_clean.loc[
        (covid_clean["New_cases"] < 0) |
        (covid_clean["New_deaths"] < 0),
        ["Date_reported", "Country", "New_cases", "New_deaths"]
    ].head(10)
)


Negative new-case records: 20
Negative new-death records: 13


,Date_reported,Country,New_cases,New_deaths
2869,2023-07-23,Australia,6495.0,-76.0
4998,2022-04-10,Belize,-2.0,8.0
6193,2021-10-17,"Bonaire, Saint Eustatius and Saba",102.0,-1.0
8236,2023-07-16,Burundi,-105.0,NaN
9156,2022-06-19,Canada,23312.0,-331.0
10173,2023-04-02,Chile,14253.0,-3432.0
10458,2024-01-14,China,1352.0,-22.0
11466,2024-08-25,Cook Islands,-1.0,NaN
14676,2020-09-13,Ecuador,-1594.0,4140.0
15487,2022-03-20,Equatorial Guinea,-98.0,NaN


In [53]:

covid_clean.to_csv(
    "COVID_Global_Impact_Cleaned.csv",
    index=False
)

print("Cleaned dataset saved!")
print("Final shape:", covid_clean.shape)


Cleaned dataset saved!
Final shape: (58560, 17)


In [54]:

from google.colab import files

files.download("COVID_Global_Impact_Cleaned.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>